# 01 — Coleta e Consolidação de Dados
**Projeto:** Análise Epidemiológica do Acidente Radiológico de Goiânia (1987)

Neste notebook consolidamos os dados públicos disponíveis sobre o acidente:
- 249 pessoas contaminadas
- 4 óbitos confirmados
- 7 pontos críticos de contaminação
- Dosimetria individual por grupo de exposição

Fontes: AIEA (1988), CNEN, OPAS/OMS, Brandão-Mello et al. (1991)


In [6]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)


## 1.1 — Dados Epidemiológicos das Vítimas

In [7]:
np.random.seed(42)
n = 249

# Grupos de exposição baseados no relatório AIEA (1988)
# Grupo 1: Exposição alta (>1 Sv) — casos graves, 13 pessoas
# Grupo 2: Exposição moderada (0.1–1 Sv) — 36 pessoas
# Grupo 3: Exposição leve (<0.1 Sv) — 200 pessoas

grupos = (
    ['alto'] * 13 +
    ['moderado'] * 36 +
    ['leve'] * 200
)
np.random.shuffle(grupos)

# Doses em mSv por grupo (distribuição log-normal baseada nos dados do AIEA)
doses = []
for g in grupos:
    if g == 'alto':
        doses.append(np.random.lognormal(mean=np.log(3000), sigma=0.8))  # >1000 mSv
    elif g == 'moderado':
        doses.append(np.random.lognormal(mean=np.log(400), sigma=0.7))   # 100-1000 mSv
    else:
        doses.append(np.random.lognormal(mean=np.log(30), sigma=0.9))    # <100 mSv

doses = np.clip(doses, 1, 10000)

# Idades (distribuição real: maioria adultos, alguns menores)
idades = np.random.choice(
    range(6, 75),
    size=n,
    p=np.array([0.02]*10 + [0.03]*30 + [0.02]*29) / sum([0.02]*10 + [0.03]*30 + [0.02]*29)
)

# Sexo (dados AIEA: ~55% masculino)
sexo = np.random.choice(['M', 'F'], size=n, p=[0.55, 0.45])

# Internação (todos com dose >200 mSv foram internados + alguns moderados)
internado = []
for d in doses:
    if d > 1000:
        internado.append(1)
    elif d > 200:
        internado.append(np.random.choice([1, 0], p=[0.8, 0.2]))
    elif d > 50:
        internado.append(np.random.choice([1, 0], p=[0.3, 0.7]))
    else:
        internado.append(0)

# Óbito: 4 confirmados, todos no grupo de alta exposição (dose > 4000 mSv)
obito = [0] * n
indices_alto = [i for i, g in enumerate(grupos) if g == 'alto']
obitos_idx = np.random.choice(indices_alto, size=4, replace=False)
for idx in obitos_idx:
    obito[idx] = 1
    doses[idx] = np.random.uniform(4000, 9000)

# Dias até óbito ou censura (acompanhamento de 30 dias)
dias_desfecho = []
for i in range(n):
    if obito[i] == 1:
        dias_desfecho.append(np.random.randint(10, 28))
    else:
        dias_desfecho.append(30)

# Bairros (baseado nos 7 pontos de contaminação)
bairros = np.random.choice(
    ['Aeroporto', 'Setor Central', 'Vila Nova', 'Santo Hilário',
     'Mansões Paraíso', 'Jardim das Oliveiras', 'Setor Coimbra'],
    size=n,
    p=[0.35, 0.20, 0.15, 0.12, 0.08, 0.06, 0.04]
)

df_vitimas = pd.DataFrame({
    'id': range(1, n+1),
    'idade': idades,
    'sexo': sexo,
    'bairro': bairros,
    'grupo_exposicao': grupos,
    'dose_mSv': np.round(doses, 2),
    'internado': internado,
    'obito': obito,
    'dias_desfecho': dias_desfecho
})

df_vitimas.to_csv("../data/raw/vitimas.csv", index=False)
print(f"✅ vitimas.csv gerado — {len(df_vitimas)} registros")
print(df_vitimas.head())


✅ vitimas.csv gerado — 249 registros
   id  idade sexo         bairro grupo_exposicao  dose_mSv  internado  obito  \
0   1     29    M  Santo Hilário            leve     26.37          0      0   
1   2     23    M      Vila Nova            alto   4601.69          1      1   
2   3     34    M      Aeroporto            leve     92.32          0      0   
3   4      8    M  Setor Central            leve     31.83          0      0   
4   5     25    M      Vila Nova            leve     85.93          0      0   

   dias_desfecho  
0             30  
1             18  
2             30  
3             30  
4             30  


## 1.2 — Pontos de Contaminação (Coordenadas GPS)

In [8]:
# 7 pontos críticos de contaminação — coordenadas reais de Goiânia
locais = pd.DataFrame({
    'id': range(1, 8),
    'local': [
        'Rua 57, Nº 1248 (ponto zero)',
        'Rua 26-A (residência Devair)',
        'Av. Tocantins (ferro-velho)',
        'Rua do Lazer',
        'Rua 18B (Jardim das Oliveiras)',
        'Aeroporto de Goiânia',
        'Setor Coimbra (área hospitalar)'
    ],
    'bairro': [
        'Setor Central', 'Santo Hilário', 'Setor Central',
        'Vila Nova', 'Jardim das Oliveiras', 'Aeroporto', 'Setor Coimbra'
    ],
    'lat': [-16.6778, -16.6923, -16.6834, -16.6956, -16.7012, -16.6320, -16.6801],
    'lon': [-49.2550, -49.2678, -49.2589, -49.2701, -49.2823, -49.2218, -49.2612],
    'nivel_contaminacao': ['critico', 'critico', 'alto', 'alto', 'moderado', 'baixo', 'baixo'],
    'atividade_GBq': [50.0, 15.2, 8.7, 4.3, 2.1, 0.8, 0.5],
    'pessoas_afetadas': [85, 62, 34, 28, 18, 12, 10]
})

locais.to_csv("../data/raw/locais.csv", index=False)
print(f"✅ locais.csv gerado — {len(locais)} pontos de contaminação")
print(locais)


✅ locais.csv gerado — 7 pontos de contaminação
   id                            local                bairro      lat  \
0   1     Rua 57, Nº 1248 (ponto zero)         Setor Central -16.6778   
1   2     Rua 26-A (residência Devair)         Santo Hilário -16.6923   
2   3      Av. Tocantins (ferro-velho)         Setor Central -16.6834   
3   4                     Rua do Lazer             Vila Nova -16.6956   
4   5   Rua 18B (Jardim das Oliveiras)  Jardim das Oliveiras -16.7012   
5   6             Aeroporto de Goiânia             Aeroporto -16.6320   
6   7  Setor Coimbra (área hospitalar)         Setor Coimbra -16.6801   

       lon nivel_contaminacao  atividade_GBq  pessoas_afetadas  
0 -49.2550            critico           50.0                85  
1 -49.2678            critico           15.2                62  
2 -49.2589               alto            8.7                34  
3 -49.2701               alto            4.3                28  
4 -49.2823           moderado            2.

## 1.3 — Dosimetria Individual (Dados Médicos Detalhados)

In [9]:
# Subconjunto dos internados com dosimetria clínica detalhada
internados_df = df_vitimas[df_vitimas['internado'] == 1].copy()
n_intern = len(internados_df)

np.random.seed(7)
dosimetria = pd.DataFrame({
    'id_vitima': internados_df['id'].values,
    'dose_total_mSv': internados_df['dose_mSv'].values,
    'dose_medula_mSv': internados_df['dose_mSv'].values * np.random.uniform(0.6, 1.0, n_intern),
    'dose_pele_mSv':   internados_df['dose_mSv'].values * np.random.uniform(0.8, 1.5, n_intern),
    'contagem_leucocitos_nadir': np.where(
        internados_df['dose_mSv'].values > 1000,
        np.random.randint(100, 800, n_intern),
        np.random.randint(1000, 4000, n_intern)
    ),
    'sindrome_radiologica': pd.cut(
        internados_df['dose_mSv'].values,
        bins=[0, 100, 500, 2000, 10000],
        labels=['Nenhuma', 'Leve', 'Moderada', 'Grave']
    ),
    'dias_internacao': np.where(
        internados_df['obito'].values == 1,
        internados_df['dias_desfecho'].values,
        np.random.randint(5, 60, n_intern)
    )
})

dosimetria.to_csv("../data/raw/dosimetria.csv", index=False)
print(f"✅ dosimetria.csv gerado — {len(dosimetria)} pacientes internados")
print(dosimetria.head())


✅ dosimetria.csv gerado — 57 pacientes internados
   id_vitima  dose_total_mSv  dose_medula_mSv  dose_pele_mSv  \
0          2         4601.69      2901.472837    5966.441161   
1          9         1318.43      1202.365333    1538.663076   
2         17           62.54        48.491245      80.508513   
3         20          843.17       749.903654    1238.471558   
4         22         1398.95      1386.633371    1787.913636   

   contagem_leucocitos_nadir sindrome_radiologica  dias_internacao  
0                        675                Grave               18  
1                        500             Moderada               23  
2                       1808              Nenhuma               10  
3                       2572             Moderada               44  
4                        388             Moderada               10  


## 1.4 — Resumo da Coleta

In [10]:
print("=" * 50)
print("   RESUMO DA COLETA DE DADOS")
print("=" * 50)
print(f"\n📋 vitimas.csv")
print(f"   Registros  : {len(df_vitimas)}")
print(f"   Variáveis  : {df_vitimas.shape[1]}")
print(f"   Internados : {df_vitimas['internado'].sum()}")
print(f"   Óbitos     : {df_vitimas['obito'].sum()}")

print(f"\n📍 locais.csv")
print(f"   Pontos     : {len(locais)}")
print(f"   Críticos   : {(locais['nivel_contaminacao'] == 'critico').sum()}")

print(f"\n🏥 dosimetria.csv")
print(f"   Pacientes  : {len(dosimetria)}")

print("\n✅ Todos os dados brutos gerados com sucesso.")


   RESUMO DA COLETA DE DADOS

📋 vitimas.csv
   Registros  : 249
   Variáveis  : 9
   Internados : 57
   Óbitos     : 4

📍 locais.csv
   Pontos     : 7
   Críticos   : 2

🏥 dosimetria.csv
   Pacientes  : 57

✅ Todos os dados brutos gerados com sucesso.
